# NLP Assignment P03
## Text Preprocessing untuk NLP

Notebook ini digunakan di Google Colab untuk mempraktikkan preprocessing teks menggunakan NLTK dan Stanza.

Pipeline yang dipelajari: normalisasi, punctuation removal, tokenisasi, stopword removal, stemming, lemmatisasi, POS tagging, dependency parsing, dan NER.

## Tujuan

1. Membuat fungsi `preprocess_text` dengan NLTK.
2. Menampilkan token, lemma, POS, dependency, dan NER dengan Stanza.
3. Mencoba pemrosesan multilingual.
4. Membandingkan hasil preprocessing NLTK dan Stanza.
5. Menjelaskan konsistensi preprocessing dan pemilihan tools.

## 1. Instalasi dan resource Google Colab

In [ ]:
%pip install -q nltk stanza scikit-learn Sastrawi flask

In [ ]:
import string
import nltk

resources = [
    ('tokenizers/punkt', 'punkt'),
    ('tokenizers/punkt_tab', 'punkt_tab'),
    ('corpora/stopwords', 'stopwords'),
    ('corpora/wordnet', 'wordnet'),
    ('corpora/omw-1.4', 'omw-1.4'),
    ('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng'),
]
for path, package in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(package, quiet=True)
print('Resource NLTK siap digunakan.')

## 2. Preprocessing dengan NLTK

Fungsi berikut mengembalikan hasil setiap tahap dalam bentuk dictionary. Contoh menggunakan teks bahasa Inggris agar resource NLTK standar dapat digunakan.

In [ ]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    normalized = text.lower()
    without_punctuation = normalized.translate(
        str.maketrans('', '', string.punctuation)
    )
    tokens = word_tokenize(without_punctuation)
    stop_words = set(stopwords.words('english'))
    filtered = [t for t in tokens if t.isalnum() and t not in stop_words]
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(t) for t in filtered]
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(t) for t in filtered]
    tagged = pos_tag(filtered)
    return {
        'original_text': text,
        'normalized_text': normalized,
        'without_punctuation': without_punctuation,
        'tokens': tokens,
        'filtered_tokens': filtered,
        'stemmed_tokens': stemmed,
        'lemmatized_tokens': lemmatized,
        'pos_tagged_tokens': tagged,
    }

text = 'Natural Language Processing is a field of artificial intelligence.'
nltk_result = preprocess_text(text)
for stage, value in nltk_result.items():
    print(f'{stage}: {value}')

## 3. Preprocessing dengan Stanza

Stanza menyediakan pipeline neural untuk tokenisasi, lemma, POS tagging, dependency parsing, dan NER. Model bahasa perlu diunduh satu kali di runtime Colab.

In [ ]:
import stanza
stanza.download('en', verbose=False)
nlp_en = stanza.Pipeline('en', processors='tokenize,mwt,pos,lemma,depparse,ner', verbose=False)
doc = nlp_en('Sajarwo Anggai teaches Advanced NLP at Pamulang University.')

for sentence in doc.sentences:
    for word in sentence.words:
        print({
            'text': word.text,
            'lemma': word.lemma,
            'upos': word.upos,
            'head': word.head,
            'dependency': word.deprel,
        })
    print('NER:', [(ent.text, ent.type) for ent in sentence.ents])

## 4. Multilingual processing

Gunakan model yang sesuai dengan bahasa dokumen. Contoh berikut memproses bahasa Inggris, Indonesia, dan Spanyol secara terpisah.

In [ ]:
languages = {
    'en': 'Natural Language Processing analyzes text.',
    'id': 'Pemrosesan bahasa alami menganalisis teks.',
    'es': 'El procesamiento del lenguaje natural analiza textos.',
}
for language in languages:
    stanza.download(language, verbose=False)

pipelines = {
    language: stanza.Pipeline(language, processors='tokenize,pos,lemma', verbose=False)
    for language in languages
}
for language, sentence in languages.items():
    doc = pipelines[language](sentence)
    tokens = [(word.text, word.lemma, word.upos) for s in doc.sentences for word in s.words]
    print(language, tokens)

## 5. Perbandingan NLTK dan Stanza

NLTK cocok untuk pembelajaran dan preprocessing klasik. Stanza menyediakan pipeline neural yang lebih lengkap untuk lemma, POS, dependency, dan NER. Hasil dapat berbeda karena tokenizer, model, tagset, dan resource bahasa yang digunakan.

In [ ]:
comparison = {
    'NLTK': {
        'strength': 'ringan, modular, mudah dipelajari',
        'output': list(nltk_result.keys()),
    },
    'Stanza': {
        'strength': 'pipeline neural dan anotasi linguistik lengkap',
        'output': ['token', 'lemma', 'POS', 'dependency', 'NER'],
    },
}
comparison

## 6. Contoh API Flask

File `tugas/app.py` pada materi menyediakan endpoint `POST /process`. Dalam tugas Colab ini, fungsi preprocessing sudah diuji langsung. API dapat dijalankan sebagai pengembangan lanjutan dengan payload:

```json
{"text": "Natural Language Processing is useful."}
```

## 7. Analisis dan kesimpulan

Jawab pertanyaan berikut berdasarkan output notebook:

1. Apa dampak lowercasing, punctuation removal, dan stopword removal?
2. Mengapa stemming dapat menghasilkan kata yang tidak baku?
3. Apa perbedaan hasil NLTK dan Stanza?
4. Mengapa model bahasa harus disesuaikan dengan dokumen?
5. Apa risiko jika preprocessing training dan testing tidak konsisten?

Tuliskan kesimpulan 3–5 paragraf dan sertakan contoh input-output yang paling penting.